In [1]:
from rsl_rl.env import VecEnv
from rsl_rl.runners import OnPolicyRunner
import torch
import numpy as np
torch.set_printoptions(precision=5, linewidth=200, sci_mode=False)

loaded_params = torch.load("calibrated_parameters.pt")
T = 60

para_true = []
for t in range(T):
    # Extract the scalar for day t, and use .view(1) to maintain compatibility 
    # with the vectorized environment (myVecEnv)
    day_params = {
        "beta1": loaded_params["beta1"][t].view(1),
        "beta2": loaded_params["beta2"][t].view(1),
        "pei":   loaded_params["pei"][t].view(1),
        "per":   loaded_params["per"][t].view(1)
    }
    para_true.append(day_params)


In [2]:
N          = 100020
state_dim  = 8
action_dim = 5
budget     = 6.
N_envs     = 1024 * 64
ΔT         = 1.0
device     = 'cpu'
usr_std = 0.01
test_num = 1
seed = 0

parameter_static = dict(N          = N,
                        B          = budget,
                        T          = T,
                        alpha      = 0.6,
                        v_max      = 0.2/14,
                        cost_ti    = 0.0977,
                        cost_ta    = 0.02,
                        cost_v     = 0.07,
                        cost_poc_0 = 0.000369,
                        cost_poc_1 = 0.001057,
                        pid        = 1.10/1000,
                        psr        = 0.7,
                        pid_plus   = 0.1221/1000,
                        pir        = 1/8,
                        prs        = 1/7)

Random_mean = {"beta2" : para_true[0]['beta2'][0].numpy().item(),
               "beta1" : para_true[0]['beta1'][0].numpy().item(),
               "pei"   : para_true[0]['pei'][0].numpy().item(),
               "per"   : para_true[0]['per'][0].numpy().item()}

space = {"beta2" : [Random_mean['beta2'],Random_mean['beta2']*usr_std],
         "beta1" : [Random_mean['beta1'], Random_mean['beta1']*usr_std],
         "pei"   : [Random_mean['pei'], Random_mean['pei']*usr_std],
         "per"   : [Random_mean['per'], Random_mean['per']*usr_std]}

Init_state = torch.tensor([[100000 / N, 20 / N, 0, 0, 0, 0, 1, 0]], 
                         dtype=torch.float32).to(torch.device(device))

In [3]:
class myVecEnv(VecEnv):
    def __init__(self, interval, n_envs = N_envs):
        self.B = parameter_static['B']  # total budget
        self.N = parameter_static['N']
        self.device = device
        self.init_state = Init_state
        self.states = self.init_state.repeat(n_envs, 1)
        self.num_envs = n_envs
        self.num_obs = state_dim
        self.num_privileged_obs = None
        self.num_actions = action_dim
        self.max_episode_length = parameter_static['T']
        
        torch.manual_seed(seed)
        self.parameter_random = {
            key: torch.empty(self.num_envs).normal_(
                interval[key][0],
                interval[key][1]
            ).to(torch.device(device))
            for key, value in Random_mean.items()
        }

        for key, value in Random_mean.items():
            indice = self.parameter_random[key] < 0
            self.parameter_random[key][indice] = -self.parameter_random[key][indice] 

        # optimization flags for pytorch JIT
        torch._C._jit_set_profiling_mode(False)
        torch._C._jit_set_profiling_executor(False)
        
        # allocate buffers
        self.privileged_obs_buf = None
        self.obs_buf = torch.zeros(self.num_envs, self.num_obs, device=self.device, dtype=torch.float)
        self.rew_buf = torch.zeros(self.num_envs, device=self.device, dtype=torch.float)
        self.reset_buf = torch.ones(self.num_envs, device=self.device, dtype=torch.long)
        self.episode_length_buf = torch.zeros(self.num_envs, device=self.device, dtype=torch.long)
        
        self.actions = torch.zeros(self.num_envs, self.num_actions, dtype=torch.float, device=self.device, requires_grad=False)
        self.episode_sums = torch.zeros(self.num_envs, dtype=torch.float, device=self.device, requires_grad=False)
        self.episode_acts = torch.zeros(self.num_envs, dtype=torch.float, device=self.device, requires_grad=False)
        self.extras = {}
    
    def para_update(self, para):
        self.parameter_random = para
        
    def step(self, actions):
        # actions clip
        soft_bound = [-0.5, 1.5]
        out_of_limits = -(actions - soft_bound[0]).clip(max=0.)
        out_of_limits += (actions - soft_bound[1]).clip(min=0.)
        scale_action = -1
        reward_action = scale_action * torch.sum(out_of_limits, dim=1).clip(max=1.).to(self.device)
        self.actions = torch.clip(actions, 0., 1.).to(self.device)

        # update actions & states
        b, p = self.states[:, -2] * self.B, self.states[:, -1]
        inv_c = self.cost_function(self.actions)
        b -= inv_c
        b = torch.clip(b, 0., self.B).to(self.device)
        mask = (b > 0).float().unsqueeze(-1)
        self.act = self.actions * mask
        inv_c = self.cost_function(self.act)
        next_state, reward_original = self.transition(self.act)
        self.extras['reward_original'] = reward_original
        self.extras['inv_cost'] = inv_c
        self.episode_length_buf += 1
        p = self.episode_length_buf / self.max_episode_length
        self.states = torch.cat([next_state / self.N, (b / self.B).unsqueeze(-1), p.unsqueeze(-1)], dim=-1)
        
        # check termination
        self.reset_buf = self.episode_length_buf >= self.max_episode_length
        
        # compute reward
        r_mean = -6584
        r_std  =  7347
        reward = (reward_original - inv_c - r_mean) / r_std
        self.rew_buf       = reward + reward_action
        self.episode_sums += reward + reward_action
        self.episode_acts += reward_action
        
        # compute resets
        env_ids = self.reset_buf.nonzero(as_tuple=False).flatten()
        self.reset_idx(env_ids)
        
        # compute observations
        return self.get_observations(), self.states, self.rew_buf, self.reset_buf, self.extras

    def reset_idx(self, env_ids):
        """Reset selected robots"""
        if len(env_ids) == 0:
            return
        
        assert len(env_ids) == self.num_envs
        # reset robot states
        self.states[env_ids] = self.init_state
        # reset buffers
        self.episode_length_buf[env_ids] = 0
        self.reset_buf[env_ids] = 1
        # fill extras
        self.extras["episode"] = {}
        self.extras["episode"]['rew_sum'] = torch.mean(self.episode_sums[env_ids]) / (self.max_episode_length * ΔT)
        self.extras["episode"]['rew_act'] = torch.mean(self.episode_acts[env_ids]) / (self.max_episode_length * ΔT)
        self.episode_sums[env_ids] = 0.
        self.episode_acts[env_ids] = 0.
    
    def reset(self):
        """ Reset all robots"""
        self.reset_idx(torch.arange(self.num_envs, device=self.device))
        return {}, {}
    
    def get_observations(self):
        o_mean = torch.tensor([[0.58712867, 0.06804276, 0.00308146, 0.25684301, 0.00261079, 0.08229331, 0.26572794, 0.51      ]], dtype=torch.float32).to(torch.device(self.device))
        o_std  = torch.tensor([[0.41484398, 0.07883003, 0.00389494, 0.292568  , 0.00394907, 0.06906768, 0.33943075, 0.28861739]], dtype=torch.float32).to(torch.device(self.device))
        self.obs_buf = (self.states - o_mean) / o_std
        return self.obs_buf
    
    def get_privileged_observations(self):
        return self.privileged_obs_buf

    def cost_function(self, action):
        states = self.states[:,:-2] * self.N
        st, et, at, it, dt, rt = states[:, 0], states[:, 1], states[:, 2], states[:, 3], states[:, 4], states[:, 5]
        u_sv, u_it, u_aq, u_iq, w = action[:, 0], action[:, 1], action[:, 2], action[:, 3], action[:, 4]

        cost_it  = it * u_it * parameter_static['cost_ti']
        cost_v   = st * u_sv * parameter_static['cost_v'] * parameter_static['v_max']
        cost_q   = (u_aq * at + u_iq * it) * parameter_static['cost_ta']
        per_poc  = ((st + et + at + rt) / self.N).pow(2) * parameter_static['cost_poc_0'] + parameter_static['cost_poc_1']
        cost_poc = (st + et + rt + at) * w * per_poc
        return cost_poc + cost_it + cost_v + cost_q

    def transition(self, action):
        states = self.states[:,:-2] * self.N
        st, et, at, it, dt, rt = states[:, 0], states[:, 1], states[:, 2], states[:, 3], states[:, 4], states[:, 5]
        u_sv, u_it, u_aq, u_iq, w = action[:, 0], action[:, 1], action[:, 2], action[:, 3], action[:, 4]
        ht = st + et + at + it + rt

        st_out   = (1 - u_sv * parameter_static['v_max']) * st * (it * (1 - u_iq) * self.parameter_random['beta1'] + (et + at * (1 - u_aq)) * self.parameter_random['beta2']) / ht
        st_out_e = st_out * parameter_static['alpha']
        st_out_i = st_out * (1 - parameter_static['alpha'])
        st_out_r = u_sv * parameter_static['v_max'] * st * parameter_static['psr']
        et_out_r = et * self.parameter_random['per']
        et_out_i = et * self.parameter_random['pei']
        et_out_a = et * w * (1 - self.parameter_random['per'] - self.parameter_random['pei'])
        at_out_i = at * self.parameter_random['pei']
        at_out_r = at * self.parameter_random['per']
        it_out_r = it * parameter_static['pir'] * u_it
        it_out_d = it * parameter_static['pid'] * (1 - u_it) + it * u_it * parameter_static['pid_plus']
        rt_out_s = rt * parameter_static['prs']
        
        stn = st - st_out_e - st_out_i - st_out_r + rt_out_s
        etn = et - et_out_r - et_out_i - et_out_a + st_out_e
        atn = at - at_out_r - at_out_i + et_out_a
        itn = it - it_out_r - it_out_d + st_out_i + et_out_i + at_out_i
        dtn = dt + it_out_d
        rtn = rt + et_out_r + at_out_r + it_out_r + st_out_r - rt_out_s

        reward = -2.6 * st_out - 100 * it_out_d

        next_state = torch.stack([stn, etn, atn, itn, dtn, rtn], dim=-1)
        return next_state, reward

In [4]:
def para_sampling(intvl, num_sample=N_envs):
    parameter_random = {
        key: torch.empty(num_sample).normal_(
        intvl[key][0],
        intvl[key][1]
        ).to(torch.device(device))
        for key, value in Random_mean.items()
    }
    for key, value in Random_mean.items():
        indice = parameter_random[key] < 0
        parameter_random[key][indice] = -parameter_random[key][indice] 
    return parameter_random

def update_space(final_indices, para_buf, space):
    para_est = {}
    for key, tensor_values in para_buf.items():
        selected_values = torch.mean(tensor_values[final_indices[0], final_indices[1]]) 
        para_values = torch.mean(tensor_values[final_indices[0], final_indices[1]], dim=0)
        update_values = space[key][0] + 0.5 * (selected_values - space[key][0])
        space[key] = [update_values , update_values  * usr_std]
        para_est[key] = para_values
    return space, para_est


def cal_err(para_value, env_para):
    gap = []
    for key, value in env_para.items():
        gap_temp = torch.mean(torch.abs(para_value[key] - env_para[key])/(env_para[key]))
        gap.append(gap_temp)
    return sum(gap)/len(gap) # 所有值都在范围内，返回 0
    
def state_estimator(env_sample, action_sim, obs_sim, state_prior, space, iter): 
    env_sample.reset()
    env_sample.states = state_prior  
    para_buf, mse_buf, obs_buf, states_buf = {}, {}, {}, {}
    for i in range(iter):
        para = para_sampling(space, test_num)
        env_sample.para_update(para)
        obs_sample, states_sample, _, _, _ = env_sample.step(action_sim.detach())  
        mse = torch.mean((obs_sim[:,2:4] - obs_sample[:,2:4]) ** 2, dim=1) 
        if i == 0:
            mse_buf = mse.unsqueeze(0)
            obs_buf = obs_sample.unsqueeze(0)
            states_buf = states_sample.unsqueeze(0)
            for key, values in para.items():
                para_buf[key] = para[key].unsqueeze(0)
        else:
            mse_buf = torch.cat((mse_buf, mse.unsqueeze(0)), dim=0)
            obs_buf = torch.cat((obs_buf, obs_sample.unsqueeze(0)),dim=0)
            states_buf = torch.cat((states_buf, states_sample.unsqueeze(0)), dim=0)
            for key, values in para.items():
                para_buf[key] = torch.cat((para_buf[key], para[key].unsqueeze(0)), dim=0)
                
     
    k = int(iter * 0.05)
    _, row_indices = torch.topk(mse_buf, k=k, dim=0, largest=False)
    column_indices = torch.arange(mse_buf.shape[1]).repeat(k, 1)
    top_indices = [row_indices, column_indices]
    obs_est = torch.mean(obs_buf[row_indices,column_indices,:], dim=0)
    state_est = torch.mean(states_buf[row_indices,column_indices,:], dim=0)
    state_est[:,-1] = state_prior[:,-1] + torch.tensor(1/T)
    return obs_est, state_est, top_indices, para_buf

In [5]:
import numpy as np
import shutil
import sys
import os.path
import math
from pyomo.environ import *
from pyomo.environ import SolverFactory


def solve_pyomo(init_state, budget, para, H):
    p_ei  = para['pei']
    p_er  = para['per']
    p_ea  = 1-p_ei-p_er
    p_ai  = p_ei
    p_ar  = p_er
    p_irr = 1/8
    alpha  = 0.6
    beta_1 = para['beta1']
    beta_2 = para['beta2']
    cost_tr = 0.0977
    p_id  = 1.1/1000
    p_idd = 0.1221/1000
    p_srr = 0.7
    p_rs = 1/7
    S = init_state[0]
    E = init_state[1]
    A = init_state[2]
    I = init_state[3]
    R = init_state[4]
    D = init_state[5]
    N = np.sum(init_state)
    K = H
    w_max  = 1.0
    tr_max = 1.0
    v_max = 0.2/14
    M = budget
    
    m = ConcreteModel()
    m.k = RangeSet(0, K)
    m.t = RangeSet(0, K-1)
    m.N = N
    m.tri = Var(m.k, bounds=(0, 1)) 
    m.wi = Var(m.k, bounds=(0, 1))  
    m.v = Var(m.k, bounds=(0, 1))
    m.aq = Var(m.k, bounds=(0, 1))
    m.iq = Var(m.k, bounds=(0, 1))
    m.x_s = Var(m.k, domain = NonNegativeReals)
    m.x_e = Var(m.k, domain = NonNegativeReals)
    m.x_a = Var(m.k, domain = NonNegativeReals)
    m.x_i = Var(m.k, domain = NonNegativeReals)
    m.x_r = Var(m.k, domain = NonNegativeReals)
    m.x_d = Var(m.k, domain = NonNegativeReals)
    
    m.xs_c = Constraint(m.t, rule=lambda m, k: m.x_s[k+1] == m.x_s[k]
                    -(beta_1*(1-m.iq[k])*m.x_i[k]+beta_2*(m.x_e[k]+(1-m.aq[k])*m.x_a[k]))
                    *(1-m.v[k]*v_max)*m.x_s[k]/(m.x_s[k]+m.x_e[k]+m.x_a[k]+m.x_i[k]+m.x_r[k])
                    -p_srr*m.v[k]*v_max*m.x_s[k])

    m.xe_c = Constraint(m.t, rule=lambda m, k: m.x_e[k+1] == m.x_e[k]
                        +alpha*(beta_1*(1-m.iq[k])*m.x_i[k]+beta_2*(m.x_e[k]+(1-m.aq[k])*m.x_a[k]))
                        *(1-m.v[k]*v_max)*m.x_s[k]/(m.x_s[k]+m.x_e[k]+m.x_a[k]+m.x_i[k]+m.x_r[k])
                        -p_ei*m.x_e[k]
                        -p_er*m.x_e[k]
                        -m.wi[k]*p_ea*m.x_e[k])
                     
    m.xa_c = Constraint(m.t, rule=lambda m, k: m.x_a[k+1] == m.x_a[k]
                        +m.wi[k]*p_ea*m.x_e[k]
                        -p_ai*m.x_a[k]
                        -p_ar*m.x_a[k])
                                        
    m.xi_c = Constraint(m.t, rule=lambda m, k: m.x_i[k+1] == m.x_i[k]
                        +(1-alpha)*(beta_1*(1-m.iq[k])*m.x_i[k]+beta_2*(m.x_e[k]+(1-m.aq[k])*m.x_a[k]))
                        *(1-m.v[k]*v_max)*m.x_s[k]/(m.x_s[k]+m.x_e[k]+m.x_a[k]+m.x_i[k]+m.x_r[k])
                        +p_ei*m.x_e[k]
                        +p_ai*m.x_a[k]
                        -m.tri[k]*p_irr*m.x_i[k]
                        -(1-m.tri[k])*m.x_i[k]*p_id
                        -m.tri[k]*m.x_i[k]*p_idd)
                                                                                       
    m.xr_c = Constraint(m.t, rule=lambda m, k: m.x_r[k+1] == m.x_r[k]
                        +m.tri[k]*p_irr*m.x_i[k]
                        +p_er*m.x_e[k]
                        +p_ar*m.x_a[k]
                        +p_srr*m.v[k]*v_max*m.x_s[k]-m.x_r[k]*p_rs)

    m.xd_c = Constraint(m.t, rule=lambda m, k: m.x_d[k+1] == m.x_d[k]
                        +(1-m.tri[k])*m.x_i[k]*p_id
                        +m.tri[k]*m.x_i[k]*p_idd)
    
    m.pc = ConstraintList()
    m.pc.add(m.x_s[0]==S)
    m.pc.add(m.x_e[0]==E)
    m.pc.add(m.x_a[0]==A)
    m.pc.add(m.x_i[0]==I)
    m.pc.add(m.x_r[0]==R)
    m.pc.add(m.x_d[0]==D)

    gamma = 0.99
    m.sumcost = sum(m.wi[k]*(0.000369*(m.wi[k]*(m.x_s[k]+m.x_a[k]+m.x_e[k]+m.x_r[k])/m.N)
                         *(m.wi[k]*(m.x_s[k]+m.x_e[k]+m.x_a[k]+m.x_r[k])/m.N) + 0.001057)
                         *(m.x_s[k]+m.x_e[k]+m.x_a[k]+m.x_r[k])
                    +0.0977*m.tri[k]*m.x_i[k]+m.x_s[k]
                    *m.v[k]*v_max*0.07 + m.x_i[k]*m.iq[k]*0.02 + m.x_a[k]*m.aq[k]*0.02 for k in m.t) 
    m.budget_c = Constraint(expr = m.sumcost <= M)

    m.suminfected = sum(2.6 * (beta_1*(1-m.iq[k])*m.x_i[k]+beta_2*(m.x_e[k]+(1-m.aq[k])*m.x_a[k]))
                        *(1-m.v[k]*v_max)*m.x_s[k]/(m.x_s[k]+m.x_e[k]+m.x_a[k]+m.x_i[k]+m.x_r[k])
                        +(m.x_d[k+1] - m.x_d[k]) * 100 for k in m.t)
    m.obj = Objective(expr = m.suminfected+m.sumcost, sense = minimize)

    solver = SolverFactory('ipopt')

    solver.solve(m,tee=False)
    k = np.array([k for k in m.k])
    tri = np.array([m.tri[k]() for k in m.k])
    wi = np.array([m.wi[k]() for k in m.k])
    vi = np.array([m.v[k]() for k in m.k])
    aq  = np.array([m.aq[k]() for k in m.k])
    iq  = np.array([m.iq[k]() for k in m.k])
    # return [vi[0], tri[0], aq[0], iq[0], wi[0]]
    return [vi, tri, aq, iq, wi]

In [6]:
torch.manual_seed(seed)
loaded_params = torch.load("calibrated_parameters.pt")
para_true = []
for t in range(T):
    # Extract the scalar for day t, and use .view(1) to maintain compatibility 
    # with the vectorized environment (myVecEnv)
    day_params = {
        "beta1": loaded_params["beta1"][t].view(1),
        "beta2": loaded_params["beta2"][t].view(1),
        "pei":   loaded_params["pei"][t].view(1),
        "per":   loaded_params["per"][t].view(1)
    }
    para_true.append(day_params)

In [7]:
env = myVecEnv(space, test_num)
env.para_update(para_true[0])
reward_history = []
act_buf = []
state_his_buf = []
death_buf = []

env_sample = myVecEnv(space, test_num)
state_est = env.states
obs_est = env.get_observations()
para_est = env.parameter_random
for t in range(T-1):
    actions = []
    for i in range(test_num):
        para={}
        for key, value in para_est.items():
            para[key] = float(para_est[key][i])
        aa = solve_pyomo(np.array(state_est)[i,:6] * N, np.array(state_est)[i,6] * budget, para, 15)  
        act = torch.tensor([[abs(aa[0][0]), abs(aa[1][0]), abs(aa[2][0]), 
                            abs(aa[3][0]), abs(aa[4][0])]])
        if i == 0:
            actions = act
        else:
            actions = torch.cat((actions, act), dim=0)
    
    obs, _, _, _, infos = env.step(actions.detach())
    reward_history.append((-infos['reward_original'] + infos['inv_cost']).tolist())
    death_buf.append((-infos['reward_original']).tolist())
    act_buf.append(env.act)
    state_his_buf.append(env.states)
    
    obs_est, state_est, top_indices, para_buf = state_estimator(env_sample, actions,
                                                                obs, state_est, space, 1024)
    state_est[:,2:4] = env.states[:,2:4]
    space, para_est = update_space(top_indices, para_buf, space) 
    if t < T - 1: env.para_update(para_true[t + 1])

/var/folders/xq/knm6gbz52n9dgfwjpz08m3w40000gn/T/ipykernel_84827/3474545024.py:18: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  aa = solve_pyomo(np.array(state_est)[i,:6] * N, np.array(state_est)[i,6] * budget, para, 15)
/var/folders/xq/knm6gbz52n9dgfwjpz08m3w40000gn/T/ipykernel_84827/3474545024.py:18: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  aa = solve_pyomo(np.array(state_est)[i,:6] * N, np.array(state_est)[i,6] * budget, para, 15)


In [8]:
import numpy as np
reward_history = np.array(reward_history)
print(f'score     : {np.sum(reward_history, axis=0)/T       }')
print(f'score.mean: {np.sum(reward_history, axis=0).mean()/T}')

score     : [567.02457838]
score.mean: 567.0245783805847


In [9]:
a = np.sum(reward_history, axis=0)/T 

In [10]:
min(a)

np.float64(567.0245783805847)

In [11]:
max(a)

np.float64(567.0245783805847)

In [12]:
np.std(a)

np.float64(0.0)

In [13]:
exp_result = {
    'reward_total': reward_history,
    'reward_peo': death_buf,
    'state': state_his_buf,
    'action': act_buf
}
torch.save(exp_result, 'MPC_exp4_real.pt')

In [14]:
run_result = torch.load('MPC_exp4_real.pt', weights_only=False)
death_mean = torch.mean(run_result['state'][-1][:,4]) * N
print(death_mean)

tensor(58.41847)


In [17]:
def infections(states, action, para, i):
    st, et, at, it, dt, rt = states[:, 0], states[:, 1], states[:, 2], states[:, 3], states[:, 4], states[:, 5]
    u_sv, u_it, u_aq, u_iq, w = action[:, 0], action[:, 1], action[:, 2], action[:, 3], action[:, 4]
    ht = st + et + at + it + rt

    st_out   = (1 - u_sv * parameter_static['v_max']) * st * (it * (1 - u_iq) * para['beta1'][i] + (et + at * (1 - u_aq)) * para['beta2'][i]) / ht
    st_out_e = st_out * parameter_static['alpha']
    st_out_i = st_out * (1 - parameter_static['alpha'])
    et_out_a = et * w * (1 - para['per'] - para['pei'])
    return et_out_a, st_out_e, st_out_i


init_state = torch.tensor([[100000, 20 , 0, 0, 0, 0]], dtype=torch.float32).to(torch.device(device))
s_i = torch.zeros(test_num)
s_a = torch.zeros(test_num)
s_t = torch.zeros(test_num)
for i in range(test_num):
    for t in range(T):
        if t == 0:
            sa_temp, se_temp, si_temp = infections(init_state, run_result['action'][t][i].unsqueeze(0), para_true[t], i)
        if t > 0 and t < T-1:
            sa_temp, se_temp, si_temp = infections(run_result['state'][t-1][i][:6].unsqueeze(0)*N, run_result['action'][t][i].unsqueeze(0), para_true[t], i)
        s_i[i] += si_temp[0]
        s_a[i] += sa_temp[0]
        s_t[i] += se_temp[0]+si_temp[0]
print([torch.mean(s_i), torch.mean(s_a), torch.mean(s_t)]) 

[tensor(4901.39014), tensor(    0.00000), tensor(12253.47656)]


In [1]:
567.0245783805847 * 0.6 / 6.8343

49.780481838425416